# ParfuML — catálogo público e modelos

Este notebook documenta e verifica o fluxo que realmente acompanha a versão pública do repositório ParfuML. Ele não pressupõe a distribuição da planilha bruta, de ofertas individuais, de URLs de compra ou de modelos serializados.

O artefato público principal é `public/perfumes.json`, com 3.176 perfumes. `precoMin` e `preco100ml` são valores observados/agregados; `ml.expectedPrice100ml`, `ml.priceOpportunityPct` e `ml.valueScore` são resultados derivados do modelo, não preços de loja.

A execução abaixo audita o conteúdo publicado. Ela não afirma que o conjunto privado original está dentro do GitHub.


## Fluxo de dados publicado

```text
public/perfumes.json (catálogo sanitizado, 3.176 registros)
        │
        ├── scripts/sanitize_public_data.py (auditoria/limpeza opcional)
        │
        └── ml/train_models.py
                ├── KNN por distância do cosseno + SVD (48 dimensões)
                ├── agrupamento hierárquico (8 perfis)
                └── HistGradientBoostingRegressor (preço esperado por 100 ml)
                        │
                        ├── public/perfumes.json enriquecido
                        └── ml/model_report.json
                                │
                                └── app/page.tsx (Next/React)
                                        └── filtros e ranking local no navegador
```

A aplicação não consulta uma API de lojas: ela lê o JSON estático exportado para a página.


In [ ]:
from pathlib import Path
import json
import re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "public/perfumes.json").exists():
    PROJECT_ROOT = next(parent for parent in Path.cwd().parents if (parent / "public/perfumes.json").exists())

PUBLIC_PATH = PROJECT_ROOT / "public/perfumes.json"
REPORT_PATH = PROJECT_ROOT / "ml/model_report.json"
with PUBLIC_PATH.open(encoding="utf-8") as handle:
    catalog = json.load(handle)

records = catalog["perfumes"]
print("Projeto:", PROJECT_ROOT)
print("Arquivo público:", PUBLIC_PATH)
print("Perfumes:", len(records))
print("Data da coleta:", catalog["meta"]["dataColeta"])
assert len(records) == 3176
assert catalog["meta"]["perfumes"] == len(records)


## O que é público e o que não é

Cada registro publicado combina dados de catálogo com agregações e resultados de modelagem. O pacote público não contém uma linha por oferta, nome de loja, SKU, estoque ou link de compra. `quantidadeLojas` é apenas uma contagem agregada.

Os preços devem ser descritos com precisão: `precoMin` e `preco100ml` são valores observados/agregados presentes no catálogo; `ml.expectedPrice100ml` é uma estimativa; `ml.priceOpportunityPct` e `ml.valueScore` são métricas derivadas.


In [ ]:
# Auditoria do contrato público: URLs e entidades de oferta não podem aparecer nos registros.
forbidden_key_pattern = re.compile(r"(url|link|href|store|seller|sku|estoque|stock|offer|oferta)", re.I)
forbidden = sorted({key for item in records for key in item if forbidden_key_pattern.search(key)})
print("Chaves proibidas encontradas:", forbidden or "nenhuma")
assert not forbidden
assert all("quantidadeLojas" in item for item in records)
assert all("precoMin" in item for item in records)

observed_min = [item["precoMin"] for item in records if item.get("precoMin") is not None]
observed_100ml = [item["preco100ml"] for item in records if item.get("preco100ml") is not None]
print("Registros com precoMin observado:", len(observed_min))
print("Registros com preco100ml observado:", len(observed_100ml))
print("Faixa precoMin: R$ %.2f a R$ %.2f" % (min(observed_min), max(observed_min)))
print("Faixa preco100ml: R$ %.2f a R$ %.2f" % (min(observed_100ml), max(observed_100ml)))


In [ ]:
sample = records[0]
public_fields = ["id", "nome", "marca", "genero", "familia", "notas", "precoMin", "preco100ml", "quantidadeLojas", "ml"]
print({field: sample.get(field) for field in public_fields})
assert "url" not in sample
assert isinstance(sample.get("quantidadeLojas"), int)


## Modelos e números publicados

Os números abaixo são lidos do relatório versionado, não recalculados a partir de uma fonte privada ausente. O treinamento usa Python/scikit-learn e grava o catálogo enriquecido e `ml/model_report.json`. O navegador usa os vetores, perfis e escores já gravados no JSON.


In [ ]:
with REPORT_PATH.open(encoding="utf-8") as handle:
    report = json.load(handle)

content = report["contentModel"]
clusters = report["clusterModel"]
price = report["priceModel"]
print({
    "features": content["features"],
    "dimensoes": content["embeddingDimensions"],
    "amostras_cluster": clusters["samples"],
    "clusters": clusters["clusters"],
    "amostras_preco": price["samples"],
    "treino": price["trainSamples"],
    "teste": price["testSamples"],
    "r2_log": price["r2Log"],
    "mae_brl_100ml": price["maeBrl100ml"],
    "mape": price["mape"],
})
assert content["embeddingDimensions"] == 48
assert clusters["clusters"] == 8
assert price["trainSamples"] + price["testSamples"] == price["samples"]


## Recomendação local e interpretação dos preços

A recomendação pode ser reproduzida sem links externos: os filtros usam os campos do JSON e o ranking usa os escores/vetores enriquecidos. O resultado abaixo é apenas uma demonstração do contrato de dados.


In [ ]:
def matches(perfume, genero=None, budget=None):
    if genero and perfume.get("genero") != genero:
        return False
    observed_price = perfume.get("precoMin")
    if budget is not None and (observed_price is None or observed_price > budget):
        return False
    return True

filtered = [item for item in records if matches(item, budget=300)]
ranked = sorted(filtered, key=lambda item: item.get("ml", {}).get("valueScore") or 0, reverse=True)
print("Candidatos até R$ 300 de precoMin observado:", len(filtered))
print([(item["marca"], item["nome"], item.get("precoMin")) for item in ranked[:5]])


In [ ]:
with_expected = [item for item in records if item.get("ml", {}).get("expectedPrice100ml") is not None]
print("Registros com estimativa expectedPrice100ml:", len(with_expected))
if with_expected:
    item = with_expected[0]
    print({
        "preco100ml_observado": item.get("preco100ml"),
        "expectedPrice100ml_estimado": item["ml"].get("expectedPrice100ml"),
        "priceOpportunityPct_derivado": item["ml"].get("priceOpportunityPct"),
    })


## Reprodução coerente com o repositório

A versão pública não pode ser reconstruída a partir de `data/base_perfumes_brasil.xlsx`, porque a planilha bruta não é distribuída neste checkout. Para verificar ou regenerar os artefatos públicos disponíveis:

```powershell
python scripts/sanitize_public_data.py public/perfumes.json public/perfumes.json
python ml/train_models.py
$env:GITHUB_PAGES = "true"
$env:NEXT_PUBLIC_BASE_PATH = "/parfuml"
npm run build:pages
npm run test:pages
```

A primeira linha é uma sanitização opcional e deve ser usada somente quando se quer regravar o JSON; a segunda pressupõe uma base de treino compatível disponível localmente. Este notebook audita os artefatos já publicados.


In [ ]:
expected_files = [
    PROJECT_ROOT / "public/perfumes.json",
    PROJECT_ROOT / "ml/model_report.json",
    PROJECT_ROOT / "ml/train_models.py",
    PROJECT_ROOT / "scripts/sanitize_public_data.py",
    PROJECT_ROOT / "app/page.tsx",
    PROJECT_ROOT / "tests/pages-export.test.mjs",
]
missing = [str(path.relative_to(PROJECT_ROOT)) for path in expected_files if not path.exists()]
print("Arquivos do fluxo presentes:", len(expected_files) - len(missing), "/", len(expected_files))
assert not missing, missing
print("Auditoria concluída: notebook alinhado ao pacote público ParfuML.")


## Limites e transparência

- A coleta original e eventuais fontes privadas não são reproduzíveis somente com este repositório público.
- Os preços são agregações observadas no momento da coleta; não representam disponibilidade atual nem oferta de uma loja específica.
- `expectedPrice100ml`, oportunidade e valor são resultados de modelagem e devem ser lidos como estimativas.
- Não há links de compra no catálogo público. A interface é local/estática e não realiza consulta de preços em tempo real.
